In [1]:
'''
Graded Challenge 6

Nama  : Mardi Wicaksana
Batch : CODA-RMT-016

Program ini dibuat untuk proses Data werehouse dari The Look E Commerce melakukan transformasi dan pembersihan data mentah ke bentuk sesuai  skema dari proses bisnis
'''

'\nGraded Challenge 6\n\nNama  : Mardi Wicaksana\nBatch : CODA-RMT-016\n\nProgram ini dibuat untuk proses Data werehouse dari The Look E Commerce melakukan transformasi dan pembersihan data mentah ke bentuk sesuai  skema dari proses bisnis\n'

In [2]:
!pip install pyspark

In [3]:
import sys
print(sys.executable)

/opt/conda/bin/python


In [4]:
pip show pyspark

Name: pyspark
Version: 4.1.1
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: Apache-2.0
Location: /usr/local/spark/python
Requires: py4j
Required-by: pyspark-connect
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pyspark

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Test") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

spark

In [7]:
from pyspark.sql.functions import col, to_timestamp, sum, count, when, isnull, trim

# Load Data

In [8]:
oi = spark.read.csv('order_items.csv', header=True, inferSchema=True)
oi.show(5)

+-----+--------+-------+----------+-----------------+----------+-------------------+-------------------+-------------------+-------------------+------------------+
|   id|order_id|user_id|product_id|inventory_item_id|    status|         created_at|         shipped_at|       delivered_at|        returned_at|        sale_price|
+-----+--------+-------+----------+-----------------+----------+-------------------+-------------------+-------------------+-------------------+------------------+
| 5600|    3865|   3080|     14235|            15132| Cancelled|2022-02-21 08:39:04|               NULL|               NULL|               NULL|0.0199999995529651|
|61912|   42753|  34114|     14235|           166989|  Complete|2026-01-18 14:31:06|2026-01-21 03:52:37|2026-01-25 14:04:37|               NULL|0.0199999995529651|
|48201|   33258|  26604|     14235|           129971|Processing|2024-09-15 06:10:30|               NULL|               NULL|               NULL|0.0199999995529651|
| 7175|    4958|

In [9]:
o = spark.read.csv('orders.csv', header=True, inferSchema=True)
o.show(5)

+--------+-------+---------+------+-------------------+-----------+----------+------------+-----------+
|order_id|user_id|   status|gender|         created_at|returned_at|shipped_at|delivered_at|num_of_item|
+--------+-------+---------+------+-------------------+-----------+----------+------------+-----------+
|      11|      7|Cancelled|     F|2019-11-23 18:13:42|       NULL|      NULL|        NULL|          1|
|      14|      8|Cancelled|     F|2025-06-26 00:14:48|       NULL|      NULL|        NULL|          1|
|      45|     32|Cancelled|     F|2020-09-29 23:00:16|       NULL|      NULL|        NULL|          1|
|      47|     32|Cancelled|     F|2021-08-10 02:15:58|       NULL|      NULL|        NULL|          1|
|      75|     57|Cancelled|     F|2025-02-14 01:13:45|       NULL|      NULL|        NULL|          3|
+--------+-------+---------+------+-------------------+-----------+----------+------------+-----------+
only showing top 5 rows


In [10]:
p = spark.read.csv('products.csv', header=True, inferSchema=True)
p.show(5)

+-----+------------------+-----------+--------------------+-----+------------------+----------+--------------------+----------------------+
|   id|              cost|   category|                name|brand|      retail_price|department|                 sku|distribution_center_id|
+-----+------------------+-----------+--------------------+-----+------------------+----------+--------------------+----------------------+
|13842| 2.518749990849756|Accessories|Low Profile Dyed ...|   MG|              6.25|     Women|EBD58B8A3F1D72F42...|                     1|
|13928|2.3383499148894105|Accessories|Low Profile Dyed ...|   MG| 5.949999809265137|     Women|2EAC42424D12436BD...|                     1|
|14115| 4.879559879379869|Accessories|Enzyme Regular So...|   MG|10.989999771118164|     Women|EE364229B2791D1EF...|                     1|
|14157| 4.648769887297898|Accessories|Enzyme Regular So...|   MG|10.989999771118164|     Women|00BD13095D06C20B1...|                     1|
|14273| 6.5079298864

In [11]:
u = spark.read.csv('users.csv', header=True, inferSchema=True)
u.show(5)

+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+----+-------+------------+------------+--------------+--------------------+--------------------+
|   id|first_name|last_name|               email|age|gender|state|      street_address|postal_code|city|country|    latitude|   longitude|traffic_source|          created_at|           user_geom|
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+----+-------+------------+------------+--------------+--------------------+--------------------+
|26566|  Danielle|    Baird|daniellebaird@exa...| 45|     F| Acre|1630 Christina Un...|  69980-000|null| Brasil|-8.065346116|-72.87094866|        Search| 2025-06-06 01:33:00|POINT(-72.8709486...|
|10845|  Michelle|    Moore|michellemoore@exa...| 52|     F| Acre| 1005 Alexandra Burg|  69980-000|null| Brasil|-8.065346116|-72.87094866|        Search| 2020-10-13 12:53:00|POINT(-72.8709486...|
|69495|   Kristen|  

# Data Transformation Process for Fact and Dimension Tables Based on Business Requirements 

### Selecting and transforming relevant columns for an e-commerce sales data warehouse using a star schema


In [12]:
fact_order_items = oi.select(
    col("id"),
    col("order_id"),
    col("user_id"),
    col("product_id"),
    to_timestamp(col("created_at")).alias("created_at"),
    col("sale_price")
)
fact_order_items.show(5)

+-----+--------+-------+----------+-------------------+------------------+
|   id|order_id|user_id|product_id|         created_at|        sale_price|
+-----+--------+-------+----------+-------------------+------------------+
| 5600|    3865|   3080|     14235|2022-02-21 08:39:04|0.0199999995529651|
|61912|   42753|  34114|     14235|2026-01-18 14:31:06|0.0199999995529651|
|48201|   33258|  26604|     14235|2024-09-15 06:10:30|0.0199999995529651|
| 7175|    4958|   3953|     14235|2025-05-03 09:55:46|0.0199999995529651|
|27496|   18960|  15179|     14235|2025-09-28 09:47:23|0.0199999995529651|
+-----+--------+-------+----------+-------------------+------------------+
only showing top 5 rows


In [13]:
dim_users = u.select(
    col("id").alias("user_id"),
    col("gender"),
    col("age"),
    col("country"),
    col("city")
)
dim_users.show(5)

+-------+------+---+-------+----+
|user_id|gender|age|country|city|
+-------+------+---+-------+----+
|  26566|     F| 45| Brasil|null|
|  10845|     F| 52| Brasil|null|
|  69495|     F| 57| Brasil|null|
|  96139|     M| 46| Brasil|null|
|  85768|     F| 58| Brasil|null|
+-------+------+---+-------+----+
only showing top 5 rows


In [14]:
dim_products = p.select(
    col("id").alias("product_id"),
    col("name"),
    col("category"),
    col("brand"),
    col("retail_price")
)
dim_products.show(5)

+----------+--------------------+-----------+-----+------------------+
|product_id|                name|   category|brand|      retail_price|
+----------+--------------------+-----------+-----+------------------+
|     13842|Low Profile Dyed ...|Accessories|   MG|              6.25|
|     13928|Low Profile Dyed ...|Accessories|   MG| 5.949999809265137|
|     14115|Enzyme Regular So...|Accessories|   MG|10.989999771118164|
|     14157|Enzyme Regular So...|Accessories|   MG|10.989999771118164|
|     14273|Washed Canvas Ivy...|Accessories|   MG|15.989999771118164|
+----------+--------------------+-----------+-----+------------------+
only showing top 5 rows


In [15]:
dim_orders = o.select(
    col("order_id"),
    col("status"),
    to_timestamp(col("created_at")).alias("created_at"),
    col("num_of_item")
)
dim_orders.show(5)

+--------+---------+-------------------+-----------+
|order_id|   status|         created_at|num_of_item|
+--------+---------+-------------------+-----------+
|      11|Cancelled|2019-11-23 18:13:42|          1|
|      14|Cancelled|2025-06-26 00:14:48|          1|
|      45|Cancelled|2020-09-29 23:00:16|          1|
|      47|Cancelled|2021-08-10 02:15:58|          1|
|      75|Cancelled|2025-02-14 01:13:45|          3|
+--------+---------+-------------------+-----------+
only showing top 5 rows


In [16]:
print("Fact Order Items")
fact_order_items.printSchema()
print("Dim Users")
dim_users.printSchema()
print("Dim Products")
dim_products.printSchema()
print("Dim Order")
dim_orders.printSchema()

Fact Order Items
root
 |-- id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- sale_price: double (nullable = true)

Dim Users
root
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)

Dim Products
root
 |-- product_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- retail_price: double (nullable = true)

Dim Order
root
 |-- order_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- num_of_item: integer (nullable = true)



### Process Checking Missing Value in each Table and column

In [17]:
fact_order_items.select([count(when(isnull(c), c)).alias(c) for c in fact_order_items.columns]).show()

+---+--------+-------+----------+----------+----------+
| id|order_id|user_id|product_id|created_at|sale_price|
+---+--------+-------+----------+----------+----------+
|  0|       0|      0|         0|         0|         0|
+---+--------+-------+----------+----------+----------+



In [18]:
dim_orders.select([count(when(isnull(c), c)).alias(c) for c in dim_orders.columns]).show()

+--------+------+----------+-----------+
|order_id|status|created_at|num_of_item|
+--------+------+----------+-----------+
|       0|     0|         0|          0|
+--------+------+----------+-----------+



In [19]:
dim_products.select([count(when(isnull(c), c)).alias(c) for c in dim_products.columns]).show()

+----------+----+--------+-----+------------+
|product_id|name|category|brand|retail_price|
+----------+----+--------+-----+------------+
|         0|   2|       0|   24|           0|
+----------+----+--------+-----+------------+



In [20]:
dim_users.select([count(when(isnull(c), c)).alias(c) for c in dim_users.columns]).show()

+-------+------+---+-------+----+
|user_id|gender|age|country|city|
+-------+------+---+-------+----+
|      0|     0|  0|      0|   0|
+-------+------+---+-------+----+



In [21]:
dim_users.select([
    count(when(col(c).cast("string").isin("", "null"), c)).alias(c) ##using cast string since we cannot detect null 
    for c in dim_users.columns
]).show()

+-------+------+---+-------+----+
|user_id|gender|age|country|city|
+-------+------+---+-------+----+
|      0|     0|  0|      0| 962|
+-------+------+---+-------+----+



after checking missing value in each dataset, its show dim_product have 2 name and 24 brand missing value, and dim_users have 962 city missing value.

# Handling Missing Value

## Handling dim_products

In [22]:
dim_products.filter(col("name").isNull() | col("brand").isNull()).show() ##showing data null on name and brand

+----------+--------------------+--------------------+---------------+------------------+
|product_id|                name|            category|          brand|      retail_price|
+----------+--------------------+--------------------+---------------+------------------+
|     12586|                NULL|           Intimates|Josie by Natori|              36.0|
|     24455|                NULL|   Outerwear & Coats|       Tru-Spec|147.99000549316406|
|       755|The Very Hungry C...|         Tops & Tees|           NULL|28.950000762939453|
|      1629|Carhartt Women's ...|Fashion Hoodies &...|           NULL|45.950000762939446|
|      8600|Women's Micro Fle...|   Outerwear & Coats|           NULL|  35.9900016784668|
|      9482|KEEN Women Bellin...|     Socks & Hosiery|           NULL|              16.0|
|     10598|JMS Comfort Lace ...|           Intimates|           NULL|16.579999923706055|
|     11389|Shadowline 36 Fla...|           Intimates|           NULL|              23.0|
|     1166

In [23]:
dim_products = dim_products.filter(col("name").isNotNull()) ##delete product data that does not have a product name because what they sell if there is no name ?

In [24]:
dim_products = dim_products.fillna({"brand": "Unknown"}) ##change brand null into Unknown

In [25]:
dim_products.select([count(when(isnull(c), c)).alias(c) for c in dim_products.columns]).show() ## data dim_products now have no missing value

+----------+----+--------+-----+------------+
|product_id|name|category|brand|retail_price|
+----------+----+--------+-----+------------+
|         0|   0|       0|    0|           0|
+----------+----+--------+-----+------------+



## Handling Fact_order_items

In [26]:
datanullproduk = [12586, 24455] ##removing product_id in fact table that have transasction using this product id

In [27]:
fact_order_items.filter(col("product_id").isin(datanullproduk)).show()

+------+--------+-------+----------+--------------------+------------------+
|    id|order_id|user_id|product_id|          created_at|        sale_price|
+------+--------+-------+----------+--------------------+------------------+
|155079|  106752|  85106|     12586| 2023-02-01 09:26:18|              36.0|
|157354|  108331|  86339|     12586| 2026-03-31 05:27:47|              36.0|
| 86819|   59893|  47651|     12586| 2025-07-02 19:04:56|              36.0|
| 90855|   62630|  49821|     12586| 2024-11-18 07:40:09|              36.0|
| 28552|   19674|  15756|     12586| 2023-11-22 17:12:26|              36.0|
| 39428|   27229|  21851|     12586|2026-04-16 02:40:...|              36.0|
|164636|  113406|  90415|     12586| 2025-06-23 21:55:24|              36.0|
|  2559|    1762|   1405|     24455| 2024-08-14 00:06:49|147.99000549316406|
|  9003|    6206|   4973|     24455| 2021-06-25 18:11:36|147.99000549316406|
| 29080|   20046|  16049|     24455| 2022-05-28 11:29:39|147.99000549316406|

In [28]:
##removing product_id in fact table that have transasction using this product id
fact_order_items = fact_order_items.filter(~col("product_id").isin(datanullproduk))

In [29]:
datanullorders = [106752, 108331, 59893, 62630, 19674, 27229, 113406, 1762, 6206, 20046, 23161, 18246, 15925, 28630] ##get data order_id that using removed product_id

In [30]:
fact_order_items = fact_order_items.filter(~col("order_id").isin(datanullorders)) ##delete order_id that having transaction using removed product_id

In [31]:
fact_order_items.filter(col("product_id").isin(datanullproduk)).show()

+---+--------+-------+----------+----------+----------+
| id|order_id|user_id|product_id|created_at|sale_price|
+---+--------+-------+----------+----------+----------+
+---+--------+-------+----------+----------+----------+



In [32]:
fact_order_items.filter(col("order_id").isin(datanullorders)).show()

+---+--------+-------+----------+----------+----------+
| id|order_id|user_id|product_id|created_at|sale_price|
+---+--------+-------+----------+----------+----------+
+---+--------+-------+----------+----------+----------+



## Handling dim_users

In [33]:
cityuser = dim_users.filter(col("city").isNull() | col("city").cast("string").isin("", "null")).count() / dim_users.count() * 100
print("Persentase Null City = ",cityuser,"%")

Persentase Null City =  0.962 %


In [34]:
##Handling missing value city null into Unknown because need full userID data
dim_users = dim_users.withColumn(
    "city",
    when(trim(col("city").cast("string")).isin("", "null"), "Unknown")
    .otherwise(col("city"))
)

In [35]:
dim_users.select([
    count(when(col(c).cast("string").isin("", "null"), c)).alias(c) ##
    for c in dim_users.columns
]).show() ## Checking again after handling the missing value and its 0 now

+-------+------+---+-------+----+
|user_id|gender|age|country|city|
+-------+------+---+-------+----+
|      0|     0|  0|      0|   0|
+-------+------+---+-------+----+



## Handling dim_orders

In [36]:
datanullorders = [
    106752, 108331, 59893, 62630, 19674, 27229, 113406,
    1762, 6206, 20046, 23161, 18246, 15925, 28630
]

dim_orders = dim_orders.filter(~col("order_id").isin(datanullorders))

## Delete specifit data null in orders related to Fact_orders_id, this order_id transaction related to product_id have no name that has been deleted

In [37]:
##Add new column total price
total_price = fact_order_items.groupBy("order_id").agg(sum("sale_price").alias("total_price"))
total_price.show(5)

+--------+------------------+
|order_id|       total_price|
+--------+------------------+
|   85321|172.14999413490295|
|   88674| 16.08999991416931|
|    7554|13.269999742507935|
|   77422| 41.67000126838684|
|  113652| 43.94000077247619|
+--------+------------------+
only showing top 5 rows


In [38]:
dim_orders = dim_orders.join(total_price, on="order_id", how="left")
dim_orders.show(5)

+--------+---------+-------------------+-----------+------------------+
|order_id|   status|         created_at|num_of_item|       total_price|
+--------+---------+-------------------+-----------+------------------+
|      11|Cancelled|2019-11-23 18:13:42|          1|19.989999771118164|
|      14|Cancelled|2025-06-26 00:14:48|          1| 79.51000213623048|
|      45|Cancelled|2020-09-29 23:00:16|          1| 79.98999786376953|
|      47|Cancelled|2021-08-10 02:15:58|          1|19.950000762939453|
|      75|Cancelled|2025-02-14 01:13:45|          3| 534.9799880981445|
+--------+---------+-------------------+-----------+------------------+
only showing top 5 rows


## Check Data Duplicated

In [39]:
dim_users.groupBy("user_id").count().filter("count > 1").show()

+-------+-----+
|user_id|count|
+-------+-----+
+-------+-----+



In [40]:
dim_products.groupBy("product_id").count().filter("count > 1").show()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [41]:
dim_orders.groupBy("order_id").count().filter("count > 1").show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [42]:
fact_order_items.groupBy("id").count().filter("count > 1").show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [43]:
##Result after checking duplicated, all dataset is safe have no duplicate 

In [44]:
fact_order_items.coalesce(1).write.csv('fact_order_items.csv', header=True, mode='overwrite')
dim_orders.coalesce(1).write.csv('dim_orders.csv', header=True, mode='overwrite')
dim_products.coalesce(1).write.csv('dim_products.csv', header=True, mode='overwrite')
dim_users.coalesce(1).write.csv('dim_users.csv', header=True, mode='overwrite')

##

## Process Connect PySpark to PGAdmin

In [45]:
data_foi = spark.read.csv('fact_order_items.csv', header=True, inferSchema=True)
data_do = spark.read.csv('dim_orders.csv', header=True, inferSchema=True)
data_dp = spark.read.csv('dim_products.csv', header=True, inferSchema=True)
data_du = spark.read.csv('dim_users.csv', header=True, inferSchema=True)

In [ ]:
# PostgreSQL JDBC Connection
postgres_url = "jdbc:postgresql://host.docker.internal/databasename"
postgres_properties = {
    "user": "postgresname",
    "password": "passpostgres",
    "driver": "org.postgresql.Driver"
}

# Write DataFrame to PostgreSQL
data_du.write.jdbc(url=postgres_url, table="dim_users", mode="append", properties=postgres_properties)
data_dp.write.jdbc(url=postgres_url, table="dim_products", mode="append", properties=postgres_properties)
data_do.write.jdbc(url=postgres_url, table="dim_orders", mode="append", properties=postgres_properties)
data_foi.write.jdbc(url=postgres_url, table="fact_order_items", mode="append", properties=postgres_properties)

### Best Selling Product

In [47]:
sql_query = """SELECT dp.product_id, dp.name, COUNT(*) AS total_sold
FROM fact_order_items foi 
join dim_products dp on foi.product_id = dp.product_id
GROUP BY dp.product_id, dp.name
ORDER BY total_sold DESC
LIMIT 10"""
query = f"({sql_query}) AS subquery"

result = spark.read.jdbc(
    url=postgres_url,
    table=query,
    properties=postgres_properties
)
print("Top Selling Product")
result.show()

Top Selling Product
+----------+--------------------+----------+
|product_id|                name|total_sold|
+----------+--------------------+----------+
|     27992|Guy Harvey Marlin...|        20|
|     27434|Speedo Men's Keep...|        20|
|     22221|Concitor Men's Dr...|        18|
|     17261|Fred Perry Men's ...|        18|
|     21326|Buffalo by David ...|        18|
|     25487| Diesel Men's Divine|        18|
|     19656|Fred Perry Men's ...|        18|
|     26266|N2N Net Bikini White|        17|
|     16628|Nautica Men's Lon...|        17|
|      2559|NIKE WOMEN'S PRO ...|        17|
+----------+--------------------+----------+



### User Activity on platform

In [48]:
sql_query = """SELECT foi.user_id, u.gender, u.age, COUNT(*) AS total_transaction, SUM(o.total_price) AS total_spent
FROM fact_order_items foi
JOIN dim_users u ON foi.user_id = u.user_id
JOIN dim_orders o ON foi.order_id = o.order_id
GROUP BY foi.user_id, u.gender, u.age
ORDER BY total_spent DESC
LIMIT 10"""
query = f"({sql_query}) AS subquery"

result = spark.read.jdbc(
    url=postgres_url,
    table=query,
    properties=postgres_properties
)
print("Identify Most Active User By gender, age,total_transaction and based on total spend")
result.show()

Identify Most Active User By gender, age,total_transaction and based on total spend
+-------+------+---+-----------------+-----------------+
|user_id|gender|age|total_transaction|      total_spent|
+-------+------+---+-----------------+-----------------+
|  50824|     M| 40|                4|6891.919998168945|
|  19041|     F| 40|                6|6180.040008544922|
|  67497|     M| 48|                8|6098.729982376099|
|  69040|     M| 63|                5|5541.549999237061|
|  32885|     M| 57|               10|5468.670003890991|
|  89366|     M| 64|                4|5453.960021972656|
|  29128|     M| 64|                8|5442.759971618652|
|  65752|     M| 15|                8|5439.500009536743|
|  85843|     M| 57|                5|5111.919998168945|
|  96174|     M| 45|                7|5037.350006103516|
+-------+------+---+-----------------+-----------------+



### Total Revenue

In [49]:
sql_query = """SELECT SUM(total_price) AS total_revenue
FROM dim_orders"""
query = f"({sql_query}) AS subquery"

result = spark.read.jdbc(
    url=postgres_url,
    table=query,
    properties=postgres_properties
)
print("Total Revenue")
result.show()

Total Revenue
+--------------------+
|       total_revenue|
+--------------------+
|1.0860127680883162E7|
+--------------------+



### Goolge Slide

https://docs.google.com/presentation/d/1O6shpgCYEgkzLa3gdb-bDX0fp8YpgdYmKeDo7_RPyYU/edit?usp=sharing